In [1]:
import pandas as pd
import numpy as np
import os
from collections import deque, defaultdict

# ==========================================
# 1. CONFIGURATION
# ==========================================
CATEGORY_DIR = "reconnaissance"
BASE_DIR = os.getcwd() 
INPUT_FILE = os.path.join(BASE_DIR, f"merged_{CATEGORY_DIR}.csv")
OUTPUT_FILE = os.path.join(BASE_DIR, f"features_{CATEGORY_DIR}.csv")

print(f"Loading merged dataset: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE)

# Sort chronologically by Last Time for the sliding window
df['Ltime'] = pd.to_numeric(df['Ltime'], errors='coerce')
df = df.dropna(subset=['Ltime']).sort_values('Ltime').reset_index(drop=True)

# ==========================================
# 2. GENERAL PURPOSE FEATURES & RATE
# ==========================================
print("Calculating General Purpose Features & Rate...")

# Feature 37: ct_state_ttl
df['state_ttl_combo'] = df['state'].astype(str) + "_" + df['sttl'].astype(str) + "_" + df['dttl'].astype(str)
df['ct_state_ttl'] = pd.factorize(df['state_ttl_combo'])[0]
df.drop('state_ttl_combo', axis=1, inplace=True)

# Map the exact data extracted in Notebook 1
df['ct_flw_http_mthd'] = df['temp_http_mthd_count'].fillna(0).astype(int)
df['is_ftp_login'] = df['temp_is_ftp_login'].fillna(0).astype(int)
df['ct_ftp_cmd'] = df['temp_ftp_cmd_count'].fillna(0).astype(int)

# Calculate 'rate' (Packets per second). Avoid division by zero.
df['rate'] = np.where(df['dur'] > 0, (df['Spkts'] + df['Dpkts']) / df['dur'], 0)

# ==========================================
# 3. CONNECTION FEATURES (41 - 47) - SLIDING WINDOW
# ==========================================
print("Calculating 100-Connection Sliding Window Features...")

window = deque()
counts_srv_src = defaultdict(int)
counts_srv_dst = defaultdict(int)
counts_dst_ltm = defaultdict(int)
counts_src_ltm = defaultdict(int)
counts_src_dport = defaultdict(int)
counts_dst_sport = defaultdict(int)
counts_dst_src = defaultdict(int)

res_srv_src = np.zeros(len(df), dtype=int)
res_srv_dst = np.zeros(len(df), dtype=int)
res_dst_ltm = np.zeros(len(df), dtype=int)
res_src_ltm = np.zeros(len(df), dtype=int)
res_src_dport = np.zeros(len(df), dtype=int)
res_dst_sport = np.zeros(len(df), dtype=int)
res_dst_src = np.zeros(len(df), dtype=int)

for row in df.itertuples():
    idx = row.Index
    srcip = str(row.srcip)
    dstip = str(row.dstip)
    sport = str(row.sport)
    dsport = str(row.dsport)
    service = str(row.service)
    
    counts_srv_src[(service, srcip)] += 1
    counts_srv_dst[(service, dstip)] += 1
    counts_dst_ltm[dstip] += 1
    counts_src_ltm[srcip] += 1
    counts_src_dport[(srcip, dsport)] += 1
    counts_dst_sport[(dstip, sport)] += 1
    counts_dst_src[(dstip, srcip)] += 1
    
    window.append((service, srcip, dstip, sport, dsport))
    
    if len(window) > 100:
        old_svc, old_src, old_dst, old_sport, old_dport = window.popleft()
        counts_srv_src[(old_svc, old_src)] -= 1
        counts_srv_dst[(old_svc, old_dst)] -= 1
        counts_dst_ltm[old_dst] -= 1
        counts_src_ltm[old_src] -= 1
        counts_src_dport[(old_src, old_dport)] -= 1
        counts_dst_sport[(old_dst, old_sport)] -= 1
        counts_dst_src[(old_dst, old_src)] -= 1

    res_srv_src[idx] = counts_srv_src[(service, srcip)]
    res_srv_dst[idx] = counts_srv_dst[(service, dstip)]
    res_dst_ltm[idx] = counts_dst_ltm[dstip]
    res_src_ltm[idx] = counts_src_ltm[srcip]
    res_src_dport[idx] = counts_src_dport[(srcip, dsport)]
    res_dst_sport[idx] = counts_dst_sport[(dstip, sport)]
    res_dst_src[idx] = counts_dst_src[(dstip, srcip)]

df['ct_srv_src'] = res_srv_src
df['ct_srv_dst'] = res_srv_dst
df['ct_dst_ltm'] = res_dst_ltm
df['ct_src_ltm'] = res_src_ltm
df['ct_src_dport_ltm'] = res_src_dport
df['ct_dst_sport_ltm'] = res_dst_sport
df['ct_dst_src_ltm'] = res_dst_src

# ==========================================
# 4. SANITIZATION, RENAMING & CLEANUP
# ==========================================
print("Sanitizing values and applying final UNSW formatting...")

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

string_cols = df.select_dtypes(include=['object']).columns
df[string_cols] = df[string_cols].fillna('-')

# Add the sequential ID column
df['id'] = range(1, len(df) + 1)

# Rename columns to strictly match your requested list
rename_map = {
    'Sload': 'sload',
    'Dload': 'dload',
    'Spkts': 'spkts',
    'Dpkts': 'dpkts',
    'Sjit': 'sjit',
    'Djit': 'djit',
    'Sintpkt': 'sinpkt',
    'Dintpkt': 'dinpkt',
    'smeansz': 'smean',
    'dmeansz': 'dmean',
    'res_bdy_len': 'response_body_len',
    'Stime': 'stime',
    'Ltime': 'ltime',
    'Label': 'label'
}
df.rename(columns=rename_map, inplace=True)

df['label'] = df['label'].astype(int)
df['attack_cat'] = df['attack_cat'].astype(str)

# Reordering columns
final_columns = [
    'id', 'srcip', 'sport', 'dstip', 'dsport', 'dur', 'proto', 'service', 'state', 
    'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 
    'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 
    'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 
    'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 
    'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'stime', 'ltime', 'attack_cat', 'label'
]

# Drop temporary and unused columns naturally by slicing
df = df[final_columns]

# ==========================================
# 5. FINAL EXPORT
# ==========================================
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
print(f"Feature engineering complete! Ready for ML training. Saved to {OUTPUT_FILE}")

Loading merged dataset: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance\merged_reconnaissance.csv
Calculating General Purpose Features & Rate...
Calculating 100-Connection Sliding Window Features...
Sanitizing values and applying final UNSW formatting...
Feature engineering complete! Ready for ML training. Saved to C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\reconnaissance\features_reconnaissance.csv
